In [78]:
import os
from dotenv import load_dotenv
from bardapi import Bard
import anthropic
from IPython.display import Markdown, display, update_display
import json
import google.generativeai as genai
import gradio as gr

In [53]:
# Load biến môi trường từ file .env
load_dotenv(override=True)

# Lấy API key
bard_api_key = os.getenv('BARD_API_KEY')

# Gán key và khởi tạo Bard nếu có
if bard_api_key:
    os.environ['_BARD_API_KEY'] = bard_api_key
    bard = Bard()
    print(f"Bard API Key loaded and begins with: {bard_api_key[:8]}")
else:
    print("Bard API Key not set")


genai.configure(api_key=bard_api_key)
model = genai.GenerativeModel("gemini-1.5-pro")

Bard API Key loaded and begins with: AIzaSyCO


In [55]:
system_message = "You are a helpful assistant for an Airline called FlightAI. "
system_message += "Give short, courteous answers, no more than 1 sentence. "
system_message += "Always be accurate. If you don't know the answer, say so."

In [64]:
system_message = "Bạn là một trợ lý AI hữu ích."

def chat(message, history):
    prompt = f"{system_message}\n"
    
    for msg in history:
        role = msg['role'].capitalize()
        content = msg['content']
        if role == 'User':
            prompt += f"User: {content}\n"
        elif role == 'Assistant':
            prompt += f"Assistant: {content}\n"
        else:
            prompt += f"System: {content}\n"  # Nếu là hệ thống (nếu cần thiết)

    prompt += f"User: {message}\nAssistant:"

    model = genai.GenerativeModel("gemini-1.5-pro")
    stream = model.generate_content(prompt, stream=True)

    response = ""
    for chunk in stream:
        response += chunk.text or ''
        yield response

# Tạo giao diện Gradio
gr.ChatInterface(fn=chat, type="messages").launch()

* Running on local URL:  http://127.0.0.1:7899

To create a public link, set `share=True` in `launch()`.


In [82]:
ticket_prices = {"london": "$799", "paris": "$899", "tokyo": "$1400", "berlin": "$499"}

# Hàm để lấy giá vé
def get_ticket_price(destination_city):
    print(f"Tool get_ticket_price called for {destination_city}")
    city = destination_city.lower()
    return ticket_prices.get(city, "Unknown")
    
get_ticket_price("Berlin")

Tool get_ticket_price called for Berlin


'$499'

In [68]:
price_function = {
    "name": "get_ticket_price",
    "description": "Get the price of a return ticket to the destination city. Call this whenever you need to know the ticket price, for example when a customer asks 'How much is a ticket to this city'",
    "parameters": {
        "type": "object",
        "properties": {
            "destination_city": {
                "type": "string",
                "description": "The city that the customer wants to travel to",
            },
        },
        "required": ["destination_city"],
        "additionalProperties": False
    }
}
tools = [{"type": "function", "function": price_function}]

In [88]:
def chat1(message, history):
    system_message = "Bạn là một trợ lý AI hữu ích."
    
    prompt = f"{system_message}\n"
    for msg in history:
        role = msg['role'].capitalize()
        content = msg['content']
        if role == 'User':
            prompt += f"User: {content}\n"
        elif role == 'Assistant':
            prompt += f"Assistant: {content}\n"
        else:
            prompt += f"System: {content}\n"

    prompt += f"User: {message}\nAssistant:"
    stream = model.generate_content(prompt, stream=True)
    response = ""
    for chunk in stream:
        response += chunk.text or ''
        yield response
    
    if "tool_calls" in response:
        message = response['choices'][0]['message']
        response, city = handle_tool_call(message)  # Hàm xử lý công cụ được gọi
        
        messages = history + [{"role": "user", "content": message}]
        messages.append({"role": "assistant", "content": response['content']})
        
        response = model.generate_content("\n".join([msg['content'] for msg in messages]), stream=True)
    
    return response['choices'][0]['message']['content']


In [84]:
def handle_tool_call(message):
    if 'content' in message and isinstance(message['content'], dict) and 'destination_city' in message['content']:
        city = message['content']['destination_city']
        price = get_ticket_price(city)
        
        response = {
            "role": "assistant",
            "content": f"The ticket price to {city} is {price}.",
        }
        return response, city
    else:
        response = {
            "role": "assistant",
            "content": "Sorry, I couldn't find the destination city in the message."
        }
        return response, None


In [90]:
gr.ChatInterface(fn=chat1, type="messages").launch()


* Running on local URL:  http://127.0.0.1:7904

To create a public link, set `share=True` in `launch()`.
